### Muntatge de Google Drive
Munta Google Drive al directori indicat per poder accedir a datasets, checkpoints i resultats des de Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive1')

Mounted at /content/drive1


In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import torchvision
import matplotlib.pyplot as plt

from tqdm import tqdm
from matplotlib import gridspec
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

### Configuració d’avaluació massiva
Definició de rutes de models i resultats, configuració del dispositiu (GPU/CPU), llindars d’inferència i criteris mínims (*early-stop*) per filtrar models durant l’avaluació.


In [ ]:
MODELS_DIR = "/content/drive1/MyDrive/ML/DATASETS/NODE21/MODELS1"
OUTPUT_DIR = "/content/drive1/MyDrive/ML/DATASETS/NODE21/RESULTS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SCORE_THRESH = 0.01

FP_LEVELS = (0.25, 0.5, 1, 2, 4, 8)

EARLY_STOP = {
    "min_images": 40,
    "min_sensitivity": 0.15,
    "max_fp_per_image": 20.0,
    "min_mean_iou": 0.05
}

### Rutes del dataset (Drive1)
Definició de les rutes cap a les imatges i el fitxer de metadades dins el Drive muntat a `/content/drive1`.


In [ ]:
BASE_PATH = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data"

PATH_IMAGES = f"{BASE_PATH}/images"
PATH_METADATA = f"{BASE_PATH}/metadata_augmented.csv"
PATH_METADATA = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata_augmented_def2.csv"
#PATH_METADATA = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata.csv"

print(PATH_IMAGES)
print(PATH_METADATA)

/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/images
/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata_augmented_def2.csv


### Dataset de detecció NODE21 (PNG + 3 canals)
Classe `Dataset` per a detecció d’objectes: carrega imatges PNG, les normalitza, crea una entrada de 3 canals i construeix el diccionari `target` amb les bounding boxes i etiquetes per a TorchVision.


In [ ]:
import os
import torch
from torch.utils.data import Dataset
import numpy as np
import cv2

class Node21DetectionDatasetPNG(Dataset):
    def __init__(self, df):
        # Hacemos una copia para no modificar el df original
        self.df = df.copy()

        # Filtro: solo filas con la ruta válida
        self.df = self.df[self.df["file_path"].apply(os.path.exists)]

        # Lista de indices del df, en su orden natural (MUY IMPORTANTE)
        self.indices = self.df.index.tolist()


    def __len__(self):
        return len(self.indices)


    def __getitem__(self, idx):
        # Obtener el índice REAL en el DataFrame
        real_idx = self.indices[idx]

        # Extraer toda la fila para esa imagen
        row = self.df.loc[real_idx]
        path = row["file_path"]

        # ===============================
        # 1. Cargar imagen PNG
        # ===============================
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = img.astype(np.float32)

        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        # Convertir a 3 canales
        img3 = np.stack([img, img, img], axis=0)

        # ===============================
        # 2. Obtener TODAS las anotaciones de esta imagen
        # ===============================
        rows = self.df[self.df["file_path"] == path]

        boxes = []
        labels = []

        for _, r in rows.iterrows():
            if r["label"] == 1:
                x1 = float(r["x"])
                y1 = float(r["y"])
                x2 = x1 + float(r["width"])
                y2 = y1 + float(r["height"])
                boxes.append([x1, y1, x2, y2])
                labels.append(1)

        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "area": area,
            "iscrowd": iscrowd,
            "image_id": torch.tensor([real_idx])
        }

        return torch.tensor(img3, dtype=torch.float32), target

### Preparació de DataLoaders (detecció)
Definició del `collate_fn` per a detecció (llistes d’imatges i targets), divisió *train/val* i creació dels `DataLoader` per entrenament i validació.


In [ ]:
# 1. Collate
def detection_collate(batch):
    imgs = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    return imgs, targets



# 2. Split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 3. Datasets
train_dataset = Node21DetectionDatasetPNG(train_df)
val_dataset   = Node21DetectionDatasetPNG(val_df)

# 4. Dataloaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=detection_collate,
                          num_workers=2, pin_memory=True)

val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False,
                          collate_fn=detection_collate,
                          num_workers=2, pin_memory=True)

### Càrrega i preparació del metadata
Carrega el CSV de metadades, genera la ruta real de cada imatge PNG i mostra un resum del desbalanceig entre casos negatius i positius (nòdul vs no nòdul).


In [ ]:
import pandas as pd
import os

df = pd.read_csv(PATH_METADATA)

df["file_path"] = df["img_name"].apply(
    lambda x: f"{PATH_IMAGES}/{x.replace('.mha', '.png')}"
)

df.head()
num_neg = (df["label"] == 0).sum()
num_pos = (df["label"] == 1).sum()
print("Negativos:", num_neg)
print("Positivos:", num_pos)

Negativos: 3748
Positivos: 4427


### Identificació del tipus de model segons el checkpoint
Funció auxiliar per decidir quin *builder* (arquitectura) s’ha d’utilitzar en funció del nom del fitxer del checkpoint (p. ex. EfficientNet vs ResNet).


In [ ]:
def get_model_builder(checkpoint_name):
    if "efficientnet" in checkpoint_name.lower():
        return "frcnn_efficientnet"
    else:
        return "frcnn_resnet"

### Backbone EfficientNet-B0 personalitzat
Construcció d’un backbone basat en EfficientNet-B0 (sense pesos preentrenats) i preparació de la sortida (`out_channels=1280`) per integrar-lo dins un detector.


In [ ]:
from torchvision.models import efficientnet_b0

def build_my_efficientnet_backbone():
    model = efficientnet_b0(weights=None)
    backbone = model.features
    backbone.out_channels = 1280
    return backbone

### Faster R-CNN amb backbone EfficientNet
Es construeix un detector Faster R-CNN utilitzant EfficientNet com a backbone, definint un `AnchorGenerator` simple (1 escala i 3 ratios) i configurant el model per a 2 classes (fons + nòdul).


In [ ]:
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator

def build_frcnn_efficientnet(backbone):
    backbone.out_channels = 1280

    anchor_generator = AnchorGenerator(
        sizes=((128,),),              # 1 escala
        aspect_ratios=((0.5, 1.0, 2.0),)  # 3 ratios
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=2,
        rpn_anchor_generator=anchor_generator
    )
    return model


### Faster R-CNN amb ResNet50 + FPN
Es crea un model Faster R-CNN amb backbone ResNet50 i FPN des de zero (sense pesos preentrenats) i es reemplaça el *head* de classificació/regressió per adaptar-lo a 2 classes (fons + nòdul).



In [ ]:
def build_frcnn_resnet():
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    return model

### Construcció del model (Faster R-CNN ResNet50 + FPN)
Aquesta funció crea un model **Faster R-CNN** amb backbone **ResNet50 + FPN** sense pesos preentrenats i substitueix el *head* final per adaptar-lo al nombre de classes indicat (per defecte: **2** → fons + nòdul).


In [ ]:
def build_model(num_classes=2):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features, num_classes
    )

    return model

### Càrrega segura d’un checkpoint (compatible o descartat)
Aquesta funció intenta carregar un *checkpoint* dins del model.  
Si falten pesos o en sobren (model incompatible) o hi ha un error, retorna **False** i mostra el motiu.  
Si tot encaixa correctament, retorna **True**.


In [ ]:
def load_checkpoint_safe(model, path, device):
    try:
        state = torch.load(path, map_location=device)
        missing, unexpected = model.load_state_dict(state, strict=False)

        if missing or unexpected:
            print("   Incompatible")
            return False

        return True
    except Exception as e:
        print("   Error:", e)
        return False

### Funció `compute_iou(a, b)`
Calcula la **IoU (Intersection over Union)** entre dues *bounding boxes* `a` i `b` en format:

- `a = [x1, y1, x2, y2]`
- `b = [x1, y1, x2, y2]`

.


In [ ]:
def compute_iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (a[2] - a[0]) * (a[3] - a[1])
    areaB = (b[2] - b[0]) * (b[3] - b[1])

    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

### Detecció + recollida de prediccions amb *early stop*
Aquesta funció recorre tot el `dataset` fent inferència amb el model de detecció i guardant, per cada imatge, les **GT boxes**, les **pred boxes** i els **scores** (filtrats per `score_thresh`).  
A cada imatge calcula *matching* amb IoU ≥ **0.2** per comptar **TP**, **FP** i acumular **IoU mitjana**.  
Quan ja s’han processat almenys `min_images`, aplica un criteri d’aturada primerenca: si la **sensibilitat** és massa baixa, els **FP/imatge** massa alts, o la **IoU mitjana** massa baixa, **para abans** i retorna **(None, None)**.  
Si el model és acceptable, continua fins al final i retorna **(detections, gt_count)**.


In [ ]:
def collect_detections_early_stop(
    model, dataset, device, score_thresh, early_cfg
):
    model.eval()

    detections = []
    gt_count = tp = fp = iou_sum = iou_n = 0
    n_images = len(dataset)

    with torch.no_grad():
        for i in range(n_images):
            img, target = dataset[i]

            # IMAGEN A GPU
            img = img.to(device, non_blocking=True)

            gt_boxes = target["boxes"].cpu().numpy()
            gt_count += len(gt_boxes)

            # INFERENCIA EN CUDA
            pred = model([img])[0]

            boxes = pred["boxes"].detach().cpu().numpy()
            scores = pred["scores"].detach().cpu().numpy()

            keep = scores >= score_thresh
            boxes = boxes[keep]
            scores = scores[keep]

            # matching CPU
            matched = set()
            for pb in boxes:
                best_iou, best_gt = 0.0, -1
                for j, gt in enumerate(gt_boxes):
                    if j in matched:
                        continue
                    iou = compute_iou(pb, gt)
                    if iou > best_iou:
                        best_iou, best_gt = iou, j

                if best_iou >= 0.2:
                    tp += 1
                    matched.add(best_gt)
                    iou_sum += best_iou
                    iou_n += 1
                else:
                    fp += 1

            detections.append({
                "gt_boxes": gt_boxes,
                "pred_boxes": boxes,
                "scores": scores
            })


            if i % 50 == 0 or i == n_images - 1:
                print(
                    f"    Inferencia: {i+1}/{n_images} imágenes",
                    end="\r"
                )

            # -------- EARLY STOP --------
            if i + 1 >= early_cfg["min_images"]:
                sens = tp / (gt_count + 1e-8)
                fp_img = fp / (i + 1)
                mean_iou = iou_sum / iou_n if iou_n else 0.0

                if (
                    sens < early_cfg["min_sensitivity"]
                    or fp_img > early_cfg["max_fp_per_image"]
                    or mean_iou < early_cfg["min_mean_iou"]
                ):
                    print(
                        f"\nEarly stop @ {i+1} | "
                        f"Sens={sens:.3f} | "
                        f"FP/img={fp_img:.1f} | "
                        f"IoU={mean_iou:.3f}"
                    )
                    return None, None

    print()
    return detections, gt_count




### Càlcul de la corba FROC (FP/imatge vs Sensibilitat)
Aquesta funció calcula la **corba FROC** a partir d’un conjunt de deteccions (`detections`).  
Per cada imatge, ordena les prediccions per **score** i fa *matching* amb les caixes de veritat-terra (**GT**) usant un llindar d’IoU (`iou_thresh`, per defecte **0.2**).  
Cada predicció es classifica com a **TP** si encaixa amb una GT no utilitzada, o com a **FP** si no arriba al llindar.  
Finalment, acumula els TP i FP globalment (ordenats per score) i retorna:
- **fps** = falsos positius per imatge  
- **sens** = sensibilitat (TP / total de nòduls GT)


In [ ]:
def compute_froc(detections, iou_thresh=0.2):
    all_tp, all_fp, all_scores = [], [], []
    total_gt = sum(len(d["gt_boxes"]) for d in detections)
    num_images = len(detections)

    for det in detections:
        matched = set()
        boxes, scores = det["pred_boxes"], det["scores"]

        order = np.argsort(scores)[::-1]
        boxes, scores = boxes[order], scores[order]

        for i, pb in enumerate(boxes):
            best_iou, best_gt = 0.0, -1
            for j, gt in enumerate(det["gt_boxes"]):
                if j in matched:
                    continue
                iou = compute_iou(pb, gt)
                if iou > best_iou:
                    best_iou, best_gt = iou, j

            if best_iou >= iou_thresh:
                all_tp.append(1)
                all_fp.append(0)
                matched.add(best_gt)
            else:
                all_tp.append(0)
                all_fp.append(1)

            all_scores.append(scores[i])

    order = np.argsort(all_scores)[::-1]
    tp = np.cumsum(np.array(all_tp)[order])
    fp = np.cumsum(np.array(all_fp)[order])

    sens = tp / (total_gt + 1e-8)
    fps = fp / num_images

    return fps, sens


### Avaluació completa d’un model amb mètrica NODE21 i FROC
Aquesta funció avalua un model de detecció sobre un `dataset` i calcula un **NODE21 score** basat en la corba **FROC**.  

Primer fa inferència amb `collect_detections_early_stop()` (amb llindar `SCORE_THRESH` i criteris `EARLY_STOP`):  
si el rendiment inicial és massa dolent, atura l’avaluació i retorna `{"early_stopped": True}`.  

Si continua, calcula la corba **FROC** (`fps`, `sens`) amb `compute_froc()`, i després obté la **millor sensibilitat** assolida per sota de cada nivell de falsos positius per imatge definit a `FP_LEVELS`.  

Finalment, el **NODE21 score** es calcula com la mitjana d’aquestes sensibilitats, i es genera una gràfica FROC (escala logarítmica a l’eix X).


In [ ]:
def evaluate_model(model, dataset, device, name):
    model.eval()
    detections, gt_count = collect_detections_early_stop(
        model, dataset, device, SCORE_THRESH, EARLY_STOP
    )

    if detections is None:
        return {"early_stopped": True}

    fps, sens = compute_froc(detections)

    sens_at_fp = {
        fp: np.max(sens[fps <= fp]) if np.any(fps <= fp) else 0.0
        for fp in FP_LEVELS
    }

    node21 = float(np.mean(list(sens_at_fp.values())))

    # -------- Plot --------
    fig = plt.figure(figsize=(8, 6))
    plt.plot(fps, sens, lw=2)
    for fp, s in sens_at_fp.items():
        plt.scatter(fp, s)
        plt.text(fp, s, f"{s:.2f}", fontsize=9)

    plt.xscale("log")
    plt.xlabel("FP / image")
    plt.ylabel("Sensitivity")
    plt.title(name)
    plt.grid(True, which="both")
    plt.tight_layout()

    return {
        "node21_score": node21,
        "sensitivities": sens_at_fp,
        "figure": fig
    }


### Inferència automàtica del tipus de backbone a partir d’un checkpoint
Aquesta funció carrega un *checkpoint* i intenta deduir quin model s’ha utilitzat (**Faster R-CNN amb ResNet** o **Faster R-CNN amb EfficientNet**) mirant la forma dels pesos del bloc **RPN**.

En concret, cerca la capa `rpn.head.conv` dins l’`state_dict` i comprova el nombre de canals:

- Si detecta **1280 canals**, assumeix que el backbone és **EfficientNet** i retorna `"frcnn_efficientnet"`.
- Si detecta **256 canals**, assumeix que el backbone és **ResNet + FPN** i retorna `"frcnn_resnet"`.

Si no pot trobar aquesta capa o els canals no encaixen amb cap cas esperat, llença un error amb `ValueError`.


In [ ]:
def infer_builder_from_checkpoint(path, device):
    state = torch.load(path, map_location=device)
    # RPN conv: rpn.head.conv.0.0.weight
    for k, v in state.items():
        if "rpn.head.conv" in k and "weight" in k:
            in_ch = v.shape[0]
            if in_ch == 1280:
                return "frcnn_efficientnet"
            elif in_ch == 256:
                return "frcnn_resnet"
    raise ValueError("No se puede inferir el backbone del checkpoint")


### Avaluació massiva de múltiples models guardats (`.pth`)
Aquest codi recorre tots els *checkpoints* dins una carpeta i avalua automàticament cada model sobre el conjunt de validació (**val_dataset**), guardant els resultats i descartant els models dolents de forma ràpida.

Per cada fitxer `.pth` fa:

- **Detecta el tipus de model** a partir del checkpoint (ResNet o EfficientNet) amb `infer_builder_from_checkpoint()`.
- **Construeix el model corresponent** amb el *builder* adequat i el mou a GPU.
- **Carrega els pesos** amb `load_checkpoint_safe()`.  
  Si no és compatible, el model s’afegeix a `skipped` i s’omet.
- **Avalua el model** amb `evaluate_model()`, que calcula la FROC i el **NODE21 score**.  
  Si el model activa *early stop* (molt dolent), també s’omet.
- **Guarda la gràfica FROC** com a PNG dins `OUTPUT_DIR`.
- **Emmagatzema els resultats finals** dins `results` (score, temps i sensibilitats a diferents FP/img).

Al final mostra:
- Els resultats detallats per cada model vàlid
- El **temps total** invertit en l’avaluació completa


In [ ]:
results = []
skipped = []

model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith(".pth")]

start_all = time.time()

for fname in tqdm(model_files, desc="Evaluando modelos"):
    print(f"\n {fname}")

    # decidir qué arquitectura usar
    ckpt_path = os.path.join(MODELS_DIR, fname)
    builder = infer_builder_from_checkpoint(ckpt_path, DEVICE)

    if builder == "frcnn_efficientnet":
        backbone = build_my_efficientnet_backbone()
        model = build_frcnn_efficientnet(backbone).to(DEVICE)
    else:
        model = build_frcnn_resnet().to(DEVICE)

    # cargar checkpoint
    if not load_checkpoint_safe(
        model,
        os.path.join(MODELS_DIR, fname),
        DEVICE
    ):
        skipped.append(fname)
        continue

    t0 = time.time()
    out = evaluate_model(model, val_dataset, DEVICE, fname)

    if out.get("early_stopped"):
        skipped.append(fname)
        continue

    out["figure"].savefig(
        os.path.join(OUTPUT_DIR, fname.replace(".pth", "_froc.png")),
        dpi=300
    )

    results.append({
        "model": fname,
        "node21_score": out["node21_score"],
        "time_sec": round(time.time() - t0, 1),
        **{f"sens@{fp}": s for fp, s in out["sensitivities"].items()}
    })


    print("Resultados:")
    print(f"   Model: {fname}")
    print(f"   NODE21 score: {out['node21_score']:.4f}")
    print(f"   Tiempo: {round(time.time() - t0, 1)} s")

    for fp, s in out["sensitivities"].items():
        print(f"   Sens@{fp} FP/img: {s:.3f}")



print(f"\nTotal tiempo: {(time.time()-start_all)/60:.1f} min")


### Comparació final de models i exportació de resultats
Aquest fragment agafa tots els resultats recollits a `results` i genera una taula ordenada per rendiment.

Fa exactament això:

- Converteix `results` en un **DataFrame** i l’ordena per `node21_score` de major a menor.
- Desa la comparació completa en un fitxer CSV (`model_comparison_vindn2.csv`) dins `OUTPUT_DIR`.
- Mostra per pantalla quin és el **millor model** (la primera fila del DataFrame).
- Imprimeix la llista de **models descartats**, guardats dins `skipped` (incompatibles o aturats per early stop).


In [ ]:
df = pd.DataFrame(results).sort_values("node21_score", ascending=False)
df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison_vindn2.csv"), index=False)

print("\n Mejor modelo:")
print(df.iloc[0])

print("\n Modelos descartados:")
for m in skipped:
    print(" -", m)



 Mejor modelo:
model           checkpoint_3canal_frcnn_vindn_epoch_11_2025123...
node21_score                                             0.891275
time_sec                                                    187.2
sens@0.25                                                0.758242
sens@0.5                                                 0.892108
sens@1                                                   0.916084
sens@2                                                   0.927073
sens@4                                                   0.927073
sens@8                                                   0.927073
Name: 30, dtype: object

 Modelos descartados:


In [ ]:
os.listdir("/content/drive1/MyDrive/ML/DATASETS/NODE21")

['proccessed_data',
 'augmented_pos',
 'metadata_balanced.csv',
 'efficientnet_b0_multicanal.pth',
 'efficientnet_b0_multicanal_laplaciana.pth',
 'efficientnet_b0_multicanal_Unsharp.pth',
 'efficientnet_b0_multicanal_Unsharp_bo.pth',
 'original_data',
 'original_augmented_pos',
 'metadata_original_balanced.csv',
 'best_frcnn_node21_1024.pth',
 'best_frcnn_node21.pth',
 'metadata_balanced.gsheet',
 'models',
 'MODELS1',
 'RESULTS']